## **Limpando dados**

Primeiramente, é importante entendermos que esse dataset trata-se de um registro de transações, ou seja, não existem apenas vendas, mas também devoluções e compras da própria empresa. Sendo assim, para focarmos nas vendas vamos criar um csv que contenha apenas os dados referentes as vendas.

### **Eliminando trasações sem descrição**

- O primeiro passo é eliminar transaçoes que não contem descrição, pois não se sabe do que se trata e também não apresentam valores de entrada ou saida de produtos nem de valores monetários.

In [1]:
import duckdb
query = f"""
    SELECT 
        COUNT(*) 
    FROM 
        read_csv('complete_ecommerce_data.csv', encoding = 'latin-1') 
    WHERE 
        Description IS NOT NULL;
"""
resultado = duckdb.sql(query).show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       540455 │
└──────────────┘



### **Eliminando transações com Quantitidades negativas ou sem CustomerID**

- Agora precisamos eliminar transações em que a quantidade de itens são negativas. Isso pode representar, de acordo com a regra de negócios, devoluções feitas por clientes, perdas de estoque ou uma serie de outros fatores que não são relevantes para a análise referente as vendas.
- Da mesma forma, precisamos eliminar as trasações que não possuam CustomerID, por não se tratarem de vendas.

In [55]:
import duckdb
query = f"""
    SELECT 
        *
    FROM 
        read_csv('complete_ecommerce_data.csv', encoding = 'latin-1') 
    WHERE 
        Description IS NOT NULL AND Quantity > 0 AND CustomerID IS NOT NULL;
"""
resultado = duckdb.sql(query).df()
display(resultado)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom
...,...,...,...,...,...,...,...,...
397919,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680,France
397920,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680,France
397921,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680,France
397922,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680,France


In [ ]:
import duckdb
query = f"""
    SELECT 
        Description, SUM(Quantity) AS TotalQuantity
    FROM 
        resultado
    GROUP BY
        Description 
    ORDER BY
        TotalQuantity DESC
"""
resultado2 = duckdb.sql(query).df()
display(resultado2)

,Description,TotalQuantity
0,MUMMY MOUSE RED GINGHAM RIBBON,1.0
1,MIDNIGHT BLUE CRYSTAL DROP EARRINGS,1.0
2,FIRE POLISHED GLASS NECKL GOLD,1.0
3,CROCHET DOG KEYRING,1.0
4,PURPLE FRANGIPANI HAIRCLIP,1.0
...,...,...
3872,WHITE HANGING HEART T-LIGHT HOLDER,36725.0
3873,JUMBO BAG RED RETROSPOT,46181.0
3874,WORLD WAR 2 GLIDERS ASSTD DESIGNS,54415.0
3875,MEDIUM CERAMIC TOP STORAGE JAR,77916.0


### **Simplificando dados para análises**
*(Nesse ponto, alteramos o FROM para realizar a busca no resultado dos processos anteriores)*
- Para algumas análises, a descrição faz-se desnecessárias, sendo assim, vamos agrupar o pedidos em apenas uma linha, em que informa a quantidade de itens, o valor unitário médio de cada item, o valor total da operação e demais informações sobre o pedido.

- Por fim, para manter salva essa tabela simplificada, vamos gerar um CSV chamado vendas_simplificada.csv

In [34]:
query = f"""
    SELECT 
        InvoiceNo,
        InvoiceDate, 
        CustomerId,
        Country, 
        COUNT(Description) AS TotalItems, 
        SUM(Quantity) AS TotalQuantity, 
        ROUND(SUM(Quantity * UnitPrice)) AS TotalRevenue
    FROM 
        resultado 
    GROUP BY 
        InvoiceNo, InvoiceDate, CustomerId, Country  
    ORDER BY 
        InvoiceNo ASC"""

vendas_simplificada = duckdb.sql(query).df()
vendas_simplificada.to_csv('vendas_simplificada.csv', index=False)
display(vendas_simplificada)

,InvoiceNo,InvoiceDate,CustomerID,Country,TotalItems,TotalQuantity,TotalRevenue
0,536365,12/1/2010 8:26,17850,United Kingdom,7,40.0,139.0
1,536366,12/1/2010 8:28,17850,United Kingdom,2,12.0,22.0
2,536367,12/1/2010 8:34,13047,United Kingdom,12,83.0,279.0
3,536368,12/1/2010 8:34,13047,United Kingdom,4,15.0,70.0
4,536369,12/1/2010 8:35,13047,United Kingdom,1,3.0,18.0
...,...,...,...,...,...,...,...
18561,581583,12/9/2011 12:23,13777,United Kingdom,2,76.0,125.0
18562,581584,12/9/2011 12:25,13777,United Kingdom,2,120.0,141.0
18563,581585,12/9/2011 12:31,15804,United Kingdom,21,278.0,329.0
18564,581586,12/9/2011 12:49,13113,United Kingdom,4,66.0,339.0
